# Démonstration — prévision de consommation électrique à 24 h

**Question :** à une heure d’émission donnée, quelles seront les 24 puissances horaires suivantes ? Chaque prévision donne 24 valeurs en kW. Ce notebook lit les résultats déjà calculés par `scripts/run_rolling_cv.py` et `scripts/run_annual_final.py` ; le [guide complet](01_TP_prevision_electrique_guide.ipynb) explique le code et permet de refaire les expériences.

Le jeu UCI décrit un seul foyer : 2 075 259 lignes à la minute, sept mesures électriques, 34 589 heures après agrégation, dont 34 151 cibles valides. Le test historique avait déjà été consulté ; le bilan final est descriptif.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

candidates = [Path.cwd(), Path.cwd() / "electric-load-forecast", *Path.cwd().parents]
ROOT = next((p for p in candidates
             if (p / "results" / "rolling_cv_summary.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook dans le dépôt avec les fichiers results/ présents.")
RESULTS = ROOT / "results"
print("Dépôt :", ROOT)


## 1. Données et saisonnalité

Une heure est conservée si au moins 45 mesures minute sont valides. L’entraînement initial se termine le 25 novembre 2008 ; la validation couvre l’hiver, le printemps, l’été et l’automne en quatre blocs ; le test commence le 26 novembre 2009. Les graphiques suivants utilisent uniquement l’entraînement annuel.


In [ ]:
display(pd.read_csv(RESULTS / "dataset_description.csv", index_col=0).round(3))
for name in ["01_serie_distribution_profil.png", "05_correlations.png"]:
    display(Image(filename=str(RESULTS / "figures" / name)))


## 2. Comparaison sur les mêmes prévisions

Les modèles sont entraînés sur des origines horaires. Ils sont évalués à 0 h, 6 h, 12 h et 18 h ; chaque origine donne 24 horizons. La MAE est l’écart absolu moyen en kW sur tous les couples origine–horizon. Les quatre blocs sont évalués avec un entraînement croissant ; les anciens résultats 70/15/15 restent séparés.


In [ ]:
comparison_path = RESULTS / "annual_model_comparison.csv"
comparison = pd.read_csv(comparison_path if comparison_path.is_file()
                         else RESULTS / "rolling_cv_summary.csv")
display(comparison.sort_values("MAE_globale_kW").set_index("modele").round(4))
pairwise_path = RESULTS / "annual_cv_pairwise.csv"
if pairwise_path.is_file():
    print("Écart entre les deux premiers : bootstrap descriptif par semaine d’émission")
    display(pd.read_csv(pairwise_path).round(4))
folds = pd.read_csv(RESULTS / "rolling_cv.csv")
display(folds.pivot(index="bloc", columns="modele", values="MAE_kW").round(3))
display(Image(filename=str(RESULTS / "figures" / "12_rolling_cv_mae.png")))


## 3. Où les erreurs augmentent-elles ?

La saison est celle de **chaque heure prévue**, y compris lorsqu’un horizon traverse un changement de mois. Les émissions se chevauchent : ces scores sont descriptifs et les couples origine–horizon ne sont pas indépendants.


In [ ]:
cv_dirs = comparison["cv_dir"].unique() if "cv_dir" in comparison else [str(RESULTS)]
details = pd.concat([
    pd.read_csv(ROOT / directory / "rolling_cv_diagnostics.csv")
    for directory in cv_dirs
], ignore_index=True)
seasons = details[details["dimension"] == "saison cible"]
display(seasons.pivot(index="groupe", columns="modele", values="MAE_kW").round(3))
selected_model = comparison.sort_values("MAE_globale_kW").iloc[0]["modele"]
selected_details = details[details["modele"] == selected_model]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, dimension, xlabel in [
    (axes[0], "heure émission", "Heure d’émission"),
    (axes[1], "horizon", "Horizon (h)"),
]:
    view = selected_details[selected_details["dimension"] == dimension].copy()
    view["groupe"] = view["groupe"].astype(int)
    view.sort_values("groupe").plot(x="groupe", y="MAE_kW", marker="o", legend=False, ax=ax)
    ax.set(xlabel=xlabel, ylabel="MAE (kW)")
fig.suptitle(f"Validation — {selected_model}")
fig.tight_layout(); plt.show()
display(selected_details[selected_details["dimension"] == "pics"].round(3))


## 4. Bilan annuel et prévisions commentées

Le modèle retenu est celui dont la MAE globale de validation glissante est minimale parmi les expériences rassemblées. Son intervalle 90 % est construit à partir des erreurs de validation par horizon. La couverture observée sur le test ne constitue pas une garantie statistique.


In [ ]:
for filename in ["annual_test_scores.csv", "annual_test_intervals.csv"]:
    display(pd.read_csv(RESULTS / filename, index_col=0).round(3))
annual = pd.read_csv(RESULTS / "annual_test_diagnostics.csv")
display(annual[annual["dimension"] == "saison cible"].round(3))
display(Image(filename=str(RESULTS / "figures" / "13_annual_test_predictions.png")))


### À dire à l’oral

- Une MAE de 0,480 kW signifie environ 0,480 kW d’écart absolu par couple origine–horizon, et non 0,480 kWh d’énergie sur une journée.
- Comparer le profil moyen des sept jours, SARIMA, LSTM et TCN sur les **mêmes** blocs ; le TCN 24 h gagne seulement 0,002 kW sur la baseline en validation.
- Sur l’année de test, le TCN fait 0,463 kW de MAE contre 0,466 kW pour la baseline, mais 1,606 contre 1,844 kW sur les pics. L’hiver reste difficile : 87,06 % de couverture pour l’intervalle annoncé à 90 %.
- Les variables météo et les vacances scolaires n’ont pas été fournies ; les variables calendaires ne remplacent pas la température mesurée.
- Le test ancien ayant déjà été regardé, ne pas décrire cette dernière année comme un test encore vierge.
